In [1]:
# import libraries
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

/Users/xander/Desktop/AI/AI_Projects/question-cv/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load dotenv

load_dotenv(override=True)
openai = OpenAI()

In [3]:
# Pushover notifications app
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [4]:
# Setup pushover notifications
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [5]:
# Record user details
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interests from {name} with email {email} and notes {notes}")
    return {"recorded": "ok"}

In [6]:
#  Record unknown questions
def record_unkown_question(question):
    push(f"Recording {question} asked that I couldn't answer")
    return {"recorded": "ok"}

In [7]:
# Record user details tool
record_user_details_json = {
    "name": "record_user_details",
    "description": "Call this tool when a user provides their email or asks to be contacted for follow-up.",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "format": "email",
                "description": "The user's valid email address"
            },
            "name": {
                "type": "string",
                "description": "The user's name if explicitly provided"
            },
            "notes": {
                "type": "string",
                "description": "Relevant context or summary of the conversation"
            }
        },
        "required": ["email"],
        "additionalProperties": False
    }
}

In [8]:
# Record unknown questions tool
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Call this tool when a user asks a question that you cannot confidently answer or lack sufficient information to respond accurately.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The exact user question that could not be answered"
            }
        },
        "required": ["question"],
        "additionalProperties": False
    }
}

In [16]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json}
]

In [ ]:
# Function to take a list of tool calls and run them

import json

TOOL_REGISTRY = {
    "record_user_details": record_user_details,
    "record_unknown_question": record_unkown_question, 
}

def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name

        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError:
            result = {"error": "Invalid JSON arguments"}
            arguments = {}

        print(f"Tool called: {tool_name}", flush=True)

        tool_function = TOOL_REGISTRY.get(tool_name)

        if not tool_function:
            result = {"error": f"Unknown tool: {tool_name}"}
        else:
            try:
                result = tool_function(**arguments)
            except Exception as e:
                result = {
                    "error": str(e),
                    "tool": tool_name
                }

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

    return results

In [11]:
# Open and Read contents of CV in PDF file format
reader = PdfReader("data/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("data/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Alexander Karari."

In [12]:
# System prompt
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s CV, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the CV as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the CV. \
If you don't know the answer to any question, use your record_unknown_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [13]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:

        # LLM call
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)

        finish_reason = response.choices[0].finish_reason
        
        # Let LLM call a tool
         
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content

In [14]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/xander/Desktop/AI/AI_Projects/question-cv/.venv/lib/python3.13/site-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/xander/Desktop/AI/AI_Projects/question-cv/.venv/lib/python3.13/site-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/Users/xander/Desktop/AI/AI_Projects/question-cv/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 2220, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/Users/xander/Desktop/AI/AI_Projects/question-cv/.venv/lib/python3.13/site-packages/gradio/blocks.py", line 1729, in call_function
    prediction = await fn(*